# 3.3 Memory Systems for Conversational Applications

## Playground Notebook

LLMs are **fundamentally stateless** — every API call is isolated. Memory systems bridge this gap by storing, managing, and re-injecting conversation history into each prompt.

| Topic | What You'll Learn |
|-------|-------------------|
| **Why Memory Matters** | The statelessness problem and how memory solves it |
| **Buffer Memory** | Store everything — simple but overflows |
| **Window Memory** | Keep last K turns — fixed size, sharp cutoff |
| **Summary Memory** | LLM-compressed history — scales to long chats |
| **Summary Buffer Memory** | Hybrid: verbatim recent + summarized old — best default |
| **Conversational RAG** | Memory + retrieval + question reformulation |
| **Token Budget Management** | Controlling memory within context window limits |

### Memory Type Comparison

| Memory Type | Token Growth | Info Loss | LLM Overhead | Best For |
|-------------|-------------|-----------|-------------|----------|
| **Buffer** | Unbounded | None | None | Short chats, prototyping |
| **Window (K)** | Fixed | Sharp cutoff | None | Medium chats |
| **Token Buffer** | Fixed | Sharp cutoff | None | Variable-length messages |
| **Summary** | Slow growth | Moderate | 1 LLM call/turn | Long conversations |
| **Summary Buffer** | Very slow | Minimal | Occasional | **General purpose — best default** |
| **Entity** | Per-entity | Some | 2 LLM calls/turn | Entity-centric chats |
| **VectorStore** | Constant retrieval | Retrieval-dep. | Embedding/turn | Very long, topic-revisit |

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [1]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama

In [11]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.1)

# Quick connectivity test
test = llm.invoke("Say 'ready' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Response: {test.content}")

✅ Connected to Ollama | Model: llama3.2:3b
   Response: Ready


---

## 1. The Statelessness Problem

Every LLM API call is **completely isolated**. Without memory, the model forgets everything the moment you send a follow-up.

### The Memory Lifecycle

```
User Input
    ↓
Memory.load() → Conversation History
    ↓
Prompt Template
    ├── System Message (static instructions)
    ├── Conversation History (from memory)  ← MessagesPlaceholder
    └── Current User Message
    ↓
Chat Model → AI Response
    ↓
Memory.save(user_message, ai_response)
    ↓
Response to User
```

### Experiment 1A: Without Memory — The Problem

In [2]:
# Without memory — each call is isolated

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep answers to 1 sentence."),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

# Two separate calls — no connection between them
print("Turn 1:")
print(f"  User: My name is Alex.")
print(f"  AI:   {chain.invoke({'question': 'My name is Alex. Remember it.'})}")

print("\nTurn 2:")
print(f"  User: What is my name?")
print(f"  AI:   {chain.invoke({'question': 'What is my name?'})}")

print("\nThe model has NO idea what your name is — each call is isolated!")

Turn 1:
  User: My name is Alex.
  AI:   I'll remember that your name is Alex.

Turn 2:
  User: What is my name?
  AI:   I don't have any information about your name, as you haven't told me.

The model has NO idea what your name is — each call is isolated!


### Experiment 1B: With Memory — The Solution

In [3]:
# With memory — conversation history is passed each time

prompt_with_memory = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep answers to 1 sentence."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

memory_chain = prompt_with_memory | llm | StrOutputParser()

# Manual memory — a list we manage ourselves
history = []

def chat(user_input):
    """Send message, get response, save to history."""
    response = memory_chain.invoke({
        "chat_history": history,
        "question": user_input
    })
    # Save the exchange
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))
    return response

# Now it remembers!
print("Turn 1:")
print(f"  User: My name is Alex.")
print(f"  AI:   {chat('My name is Alex. Remember it.')}")

print("\nTurn 2:")
print(f"  User: What is my name?")
print(f"  AI:   {chat('What is my name?')}")

print(f"\nHistory size: {len(history)} messages")
print("Memory solved the problem by passing history with every call!")

Turn 1:
  User: My name is Alex.
  AI:   I've taken note that your name is Alex.

Turn 2:
  User: What is my name?
  AI:   Your name is Alex.

History size: 4 messages
Memory solved the problem by passing history with every call!


In [12]:
# Buffer Memory — stores everything, perfect recall

class BufferMemory:
    """Stores all messages. Simple but grows unbounded."""

    def __init__(self):
        self.messages = []

    def load(self):
        """Return all stored messages."""
        return self.messages.copy()

    def save(self, human_msg, ai_msg):
        """Store the exchange."""
        self.messages.append(HumanMessage(content=human_msg))
        self.messages.append(AIMessage(content=ai_msg))

    def clear(self):
        self.messages = []

    @property
    def size(self):
        return len(self.messages)


# Wire it up with a chain
buffer_mem = BufferMemory()

convo_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly AI tutor. Keep answers to 1-2 sentences."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

convo_chain = convo_prompt | llm | StrOutputParser()

def chat_buffer(user_input):
    response = convo_chain.invoke({
        "history": buffer_mem.load(),
        "input": user_input
    })
    buffer_mem.save(user_input, response)
    return response


# Multi-turn conversation
turns = [
    "My name is Alex and I'm learning about memory in LangChain.",
    "What is the simplest type of memory?",
    "What's the main problem with it?",
    "What's my name and what am I learning?",  # Tests recall
]

for i, msg in enumerate(turns, 1):
    print(f"\nTurn {i} (history: {buffer_mem.size} msgs):")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_buffer(msg)}")

print(f"\nFinal history: {buffer_mem.size} messages (all stored!)")
print("Buffer memory gives perfect recall but will overflow on long chats.")


Turn 1 (history: 0 msgs):
  User: My name is Alex and I'm learning about memory in LangChain.
  AI:   Hi Alex! LangChain is a powerful framework for building custom NLP pipelines, and understanding memory management is crucial to optimizing performance. What specific aspects of memory are you struggling with or would like to learn more about?

Turn 2 (history: 2 msgs):
  User: What is the simplest type of memory?
  AI:   In LangChain, the simplest type of memory is likely a `Memory` object, which is essentially a dictionary that stores key-value pairs. It's a basic, in-memory data structure used to store and retrieve values during the execution of your pipeline.

Turn 3 (history: 4 msgs):
  User: What's the main problem with it?
  AI:   The main issue with `Memory` objects is that they are stored in RAM and can lead to memory leaks if not properly managed, especially when dealing with large datasets or pipelines that run for extended periods. This can cause performance issues and even

---

## 3. Memory Type 2: Window Memory — Last K Turns

Keeps only the **last K conversation exchanges**, creating a sliding window. Fixed size, predictable token usage.

| Property | Value |
|----------|-------|
| Storage | Full list, returns last K pairs |
| Growth rate | Fixed — always K × 2 messages |
| Information loss | Moderate — older context is **lost** |
| Context window risk | Low — predictable budget |

### Choosing K

| K Value | ~Tokens | Use Case |
|---------|---------|----------|
| K = 3 | ~600 | Quick Q&A bots |
| K = 5 | ~1,000 | Standard chatbots |
| K = 10 | ~2,000 | Detailed conversations |
| K = 15 | ~3,000 | Complex support |

**Rule of thumb:** Set K so memory uses no more than 25-30% of your context window.

### The Cliff Problem

Window memory has a **sharp cutoff** — information doesn't gradually fade, it disappears entirely. If the user said their name in Turn 1 and K=3, by Turn 5 it's completely gone.

### Experiment 3A: Window Memory — Sliding Window

In [5]:
# Window Memory — keeps only the last K exchanges

class WindowMemory:
    """Stores all messages but returns only the last K exchanges."""

    def __init__(self, k=3):
        self.messages = []
        self.k = k

    def load(self):
        """Return last K exchanges (K*2 messages)."""
        return self.messages[-(self.k * 2):]

    def save(self, human_msg, ai_msg):
        self.messages.append(HumanMessage(content=human_msg))
        self.messages.append(AIMessage(content=ai_msg))
    
    # def save(self, human_msg, ai_msg):
    #     self.messages.append(HumanMessage(content=human_msg))
    #     self.messages.append(AIMessage(content=ai_msg))
        
    #     # Physically delete oldest messages if we exceed the window
    #     max_messages = self.k * 2
    #     if len(self.messages) > max_messages:
    #         self.messages = self.messages[-max_messages:]

    @property
    def total_stored(self):
        return len(self.messages)

    @property
    def window_size(self):
        return len(self.load())


window_mem = WindowMemory(k=2)  # Only keep last 2 exchanges

def chat_window(user_input):
    response = convo_chain.invoke({
        "history": window_mem.load(),
        "input": user_input
    })
    window_mem.save(user_input, response)
    return response


# Test the cliff problem
turns = [
    "My favorite color is blue. Remember that!",   # Turn 1 — will fall off
    "What is 2 + 2?",                               # Turn 2 — will fall off
    "What is the capital of France?",                # Turn 3
    "What is my favorite color?",                    # Turn 4 — Turn 1 is gone!
]

for i, msg in enumerate(turns, 1):
    window = window_mem.load()
    print(f"\nTurn {i} (stored: {window_mem.total_stored}, window sees: {window_mem.window_size}):")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_window(msg)}")

print(f"\nTotal stored: {window_mem.total_stored} messages")
print(f"Window sees:  {window_mem.window_size} messages (last {window_mem.k} exchanges)")
print("\nThe 'cliff problem': early context doesn't fade — it vanishes entirely!")


Turn 1 (stored: 0, window sees: 0):
  User: My favorite color is blue. Remember that!
  AI:   I've taken note that your favorite color is blue, I'll remember it for our conversation!

Turn 2 (stored: 2, window sees: 2):
  User: What is 2 + 2?
  AI:   The answer to 2 + 2 is 4.

Turn 3 (stored: 4, window sees: 4):
  User: What is the capital of France?
  AI:   The capital of France is Paris.

Turn 4 (stored: 6, window sees: 4):
  User: What is my favorite color?
  AI:   I don't have any information about your personal preferences, so I'm not sure what your favorite color is!

Total stored: 8 messages
Window sees:  4 messages (last 2 exchanges)

The 'cliff problem': early context doesn't fade — it vanishes entirely!


---

## 4. Memory Type 3: Token Buffer Memory — Precise Token Control

Like window memory but uses a **token count** instead of turn count. More precise for variable-length messages.

| Property | Value |
|----------|-------|
| Storage | Token-counted message buffer |
| Growth rate | Fixed — bounded by token limit |
| Information loss | Moderate — same cliff problem |
| Context window risk | Very low — exact token budget |

### Experiment 4A: Token Buffer Memory

In [9]:
# Token Buffer Memory — keeps messages that fit within a token budget
# We approximate tokens as words/0.75 (rough estimate for English text)

class TokenBufferMemory:
    """Keeps the most recent messages within a token budget."""

    def __init__(self, max_tokens=200):
        self.messages = []
        self.max_tokens = max_tokens

    def _estimate_tokens(self, text):
        """Rough token estimate: ~1 token per 4 characters."""
        return len(text) // 4

    def load(self):
        """Return most recent messages that fit within token budget."""
        result = []
        token_count = 0
        # Walk backwards from most recent
        for msg in reversed(self.messages):
            tokens = self._estimate_tokens(msg.content)
            if token_count + tokens > self.max_tokens:
                break
            result.insert(0, msg)
            token_count += tokens
        return result

    def save(self, human_msg, ai_msg):
        self.messages.append(HumanMessage(content=human_msg))
        self.messages.append(AIMessage(content=ai_msg))

    def get_token_usage(self):
        loaded = self.load()
        return sum(self._estimate_tokens(m.content) for m in loaded)


token_mem = TokenBufferMemory(max_tokens=150)

def chat_token(user_input):
    response = convo_chain.invoke({
        "history": token_mem.load(),
        "input": user_input
    })
    token_mem.save(user_input, response)
    return response


turns = [
    "My name is Alex and I work at a tech company in Chennai.",
    "I'm building a chatbot for customer support.",
    "We use Python and LangChain for the backend.",
    "What is my name?",
]

for i, msg in enumerate(turns, 1):
    print(f"\nTurn {i} (~{token_mem.get_token_usage()} tokens in memory):")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_token(msg)}")

print(f"\nToken budget: {token_mem.max_tokens}")
print(f"Tokens used:  ~{token_mem.get_token_usage()}")
print(f"Messages in window: {len(token_mem.load())} of {len(token_mem.messages)} total")


Turn 1 (~0 tokens in memory):
  User: My name is Alex and I work at a tech company in Chennai.
  AI:   Hi Alex, nice to meet you! It sounds like you're doing exciting work in the tech industry based in Chennai. How's life been treating you so far?

Turn 2 (~50 tokens in memory):
  User: I'm building a chatbot for customer support.
  AI:   That's a great project, Alex! Building a chatbot can be a fantastic way to improve customer experience and streamline support operations. What specific features or functionalities are you planning to include in your chatbot?

Turn 3 (~117 tokens in memory):
  User: We use Python and LangChain for the backend.
  AI:   LangChain is a popular library for building conversational AI pipelines, especially with Python as the primary language. Are you using any specific models like BERT or RoBERTa for text processing and understanding in your chatbot?

Turn 4 (~135 tokens in memory):
  User: What is my name?
  AI:   You didn't mention your name earlier, but 

---

## 5. Memory Type 4: Summary Memory — LLM-Compressed History

Uses an LLM to maintain a **running summary** instead of storing raw messages. Scales to very long conversations.

| Property | Value |
|----------|-------|
| Storage | Running text summary |
| Growth rate | Slow — summary grows logarithmically |
| Information loss | Some — exact wording and nuance lost |
| Computational overhead | **High — 1 LLM call per turn** |
| Context window risk | Very low |

### The Cost Tradeoff

Every turn requires an **extra LLM call** to update the summary. For 20 turns, that's 20 extra calls.

**Optimization:** Use a smaller, cheaper model for summaries while using a powerful model for conversation.

### Experiment 5A: Summary Memory

In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
llm = ChatOllama(model = "gpt-oss:20b-cloud", temperature=0.1)

In [20]:
# Summary Memory — LLM maintains a running summary

class SummaryMemory:
    """Uses LLM to maintain a running conversation summary."""

    def __init__(self, llm):
        self.summary = ""
        self.llm = llm
        self.turn_count = 0
        self._summarize_chain = (
            ChatPromptTemplate.from_messages([
                ("system",
                 "Progressively summarize the conversation. "
                 "Add to the existing summary with new information. "
                 "Keep it concise — 2-3 sentences max."),
                ("human",
                 "Current summary: {summary}\n\n"
                 "New exchange:\n"
                 "Human: {human_msg}\n"
                 "AI: {ai_msg}\n\n"
                 "Updated summary:"
                 "Dont include the heading lines i have given : current summary, new exchange, human, ai, updated summary in the final response generated")
            ])
            | llm | StrOutputParser()
        )

    def load(self):
        """Return summary as a system-level context."""
        return self.summary

    def save(self, human_msg, ai_msg):
        """Update the running summary with the new exchange."""
        self.summary = self._summarize_chain.invoke({
            "summary": self.summary or "(No prior conversation)",
            "human_msg": human_msg,
            "ai_msg": ai_msg
        })
        self.turn_count += 1


summary_mem = SummaryMemory(llm)

# Prompt that uses summary as context
summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly AI assistant. Keep answers as short as possible\n"
     "Conversation so far: {context}"),
    ("human", "{input}")
])

summary_chain = summary_prompt | llm | StrOutputParser()

def chat_summary(user_input):
    response = summary_chain.invoke({
        "context": summary_mem.load() or "(New conversation)",
        "input": user_input
    })
    summary_mem.save(user_input, response)
    return response


turns = [
    "My name is Alex and I'm a data scientist.",
    "I'm building a chatbot for my company.",
    "We need it to handle 1000+ customer queries per day.",
    "The main topic is product returns and refunds.",
    "What do you know about me and my project?",
]

for i, msg in enumerate(turns, 1):
    print(f"\nTurn {i}:")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_summary(msg)}")
    print(f"  Summary: {summary_mem.load()}...")

print(f"\n{'=' * 50}")
print("FINAL SUMMARY:")
print(f"{'=' * 50}")
print(summary_mem.load())
print(f"\nTurns: {summary_mem.turn_count}")
print("Summary grows slowly — scales to very long conversations!")


Turn 1:
  User: My name is Alex and I'm a data scientist.
  AI:   Nice to meet you, Alex! How can I help you with your data science work today?
  Summary: Alex is a data scientist who introduced themselves to the assistant. They asked how the assistant could help with their data science work....

Turn 2:
  User: I'm building a chatbot for my company.
  AI:   Sounds exciting! What’s the main goal of the chatbot (e.g., customer support, internal help, sales)? And which tech stack are you using?
  Summary: Alex, a data scientist, initially asked how the assistant could help with data science work. He is now building a chatbot for his company and is discussing its main goal—whether it’s for customer support, internal help, or sales—and the tech stack he plans to use....

Turn 3:
  User: We need it to handle 1000+ customer queries per day.
  AI:   Sure!  
- **Scale horizontally**: Deploy the bot in containers (Docker/K8s) or serverless functions (AWS Lambda, Azure Functions).  
- **Load‑ba

---

## 6. Memory Type 5: Summary Buffer Memory — The Best Default

The **hybrid approach** — combines verbatim recent messages with a summary of older messages. Widely considered the best general-purpose memory.

| Property | Value |
|----------|-------|
| Storage | Summary + recent message buffer |
| Growth rate | Very slow |
| Information loss | Minimal — exact recent + summarized old |
| Computational overhead | Medium — occasional LLM calls |

### Why This is the Best Default

| vs. Buffer | Handles long conversations without overflow |
|:-----------|:---|
| **vs. Window** | Doesn't lose old context — it's summarized, not deleted |
| **vs. Pure Summary** | Recent messages are exact (important for follow-ups) |
| **vs. Token Buffer** | Old context preserved in summary form |

The human memory pattern: you remember recent events in detail and older ones as general impressions.

### Experiment 6A: Summary Buffer Memory

In [21]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser


class SummaryBufferMemory:
    """
    Keeps a live window of messages. Once the window exceeds `max_messages`,
    the oldest `summarize_count` messages are folded into a running summary
    and dropped from the window.

    Every LLM call is fed:  summary  +  current message window.
    """

    def __init__(self, llm, max_messages=6, summarize_count=4):
        assert summarize_count < max_messages, "must keep at least one message after trimming"
        self.llm = llm
        self.max_messages = max_messages        # trigger point (human + ai combined)
        self.summarize_count = summarize_count   # how many oldest msgs to fold out
        self.messages = []                       # live window (verbatim)
        self.summary = ""                        # running summary of everything trimmed

        self._summarize_chain = (
            ChatPromptTemplate.from_messages([
                ("system",
                "You compress a conversation into a compact factual summary.\n"
                "\n"
                "RULES:\n"
                "- Output ONLY the summary text. No preamble, no 'Here is...', no labels.\n"
                "- Do NOT copy or quote the messages. Do NOT include 'AI:' or 'Human:' lines.\n"
                "- Write 2-4 short third-person sentences about the USER and their project.\n"
                "- Keep only durable facts: names, roles, goals, numbers, tools, decisions.\n"
                "- Drop pleasantries, questions the assistant asked, and suggestions.\n"
                "- Merge the new info into the existing summary; do not repeat facts.\n"
                "\n"
                "EXAMPLE\n"
                "Existing summary: (none)\n"
                "New messages:\n"
                "human: Hi, I'm Sam, a backend engineer.\n"
                "ai: Nice to meet you Sam! How can I help?\n"
                "human: I'm building an inventory API in Go.\n"
                "ai: Great, Go is well suited for that. What database?\n"
                "Updated summary: Sam is a backend engineer building an inventory API in Go."),
                ("human",
                "Existing summary:\n{summary}\n\n"
                "New messages:\n{new_messages}\n\n"
                "Updated summary:")
            ])
            | self.llm | StrOutputParser()
        )

    def load(self):
        """What the chat call needs: the summary + the current window."""
        return {
            "summary": self.summary or "(No prior conversation)",
            "recent": self.messages,
        }

    def save(self, human_msg, ai_msg):
        """Record one exchange, then trim if we crossed the limit."""
        self.messages.append(HumanMessage(content=human_msg))
        self.messages.append(AIMessage(content=ai_msg))
        self._trim_if_needed()

    def _trim_if_needed(self):
        # Fire as soon as we HIT the cap (== 6), not after we pass it.
        if len(self.messages) < self.max_messages:
            return

        overflow = self.messages[:self.summarize_count]      # oldest 4
        self.messages = self.messages[self.summarize_count:]  # keep the rest → 2

        new_msgs_text = "\n".join(f"{m.type}: {m.content}" for m in overflow)
        self.summary = self._summarize_chain.invoke({
            "summary": self.summary or "(none)",
            "new_messages": new_msgs_text,
        })

# Wiring it to the chat chain

sb_mem = SummaryBufferMemory(llm, max_messages=6, summarize_count=4)

sb_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a friendly AI assistant. Keep answers to 1-2 sentences.\n"
     "Summary of earlier conversation: {summary}"),
    MessagesPlaceholder(variable_name="recent"),
    ("human", "{input}"),
])

sb_chain = sb_prompt | llm | StrOutputParser()


def chat_summary_buffer(user_input):
    mem = sb_mem.load()                         # summary + current window
    response = sb_chain.invoke({
        "summary": mem["summary"],
        "recent": mem["recent"],
        "input": user_input,
    })
    sb_mem.save(user_input, response)           # store + auto-trim
    return response

# Test harness

turns = [
    "My name is Alex. I'm a Python developer.",
    "I'm building a customer support chatbot.",
    "We handle about 500 tickets per day.",
    "Most questions are about billing and refunds.",
    "We use LangChain with Ollama for the LLM.",
    "What do you know about me and my project?",
]

for i, msg in enumerate(turns, 1):
    mem = sb_mem.load()
    print(f"\nTurn {i} (window: {len(mem['recent'])} msgs, has summary: {bool(sb_mem.summary)}):")
    print(f"  User: {msg}")
    print(f"  AI:   {chat_summary_buffer(msg)}")

print("\n" + "=" * 50)
print("MEMORY STATE")
print("=" * 50)
print(f"Summary        : {sb_mem.summary}")
print(f"Window size    : {len(sb_mem.messages)} messages")
print(f"Summarize at   : > {sb_mem.max_messages} msgs, fold oldest {sb_mem.summarize_count}")


Turn 1 (window: 0 msgs, has summary: False):
  User: My name is Alex. I'm a Python developer.
  AI:   Nice to meet you, Alex! How can I help you with your Python projects today?

Turn 2 (window: 2 msgs, has summary: False):
  User: I'm building a customer support chatbot.
  AI:   That’s a great project! Start by defining the key intents (e.g., order status, return policy, FAQ) and use a library like Rasa or Dialogflow to handle NLP, then integrate it with your backend to fetch real‑time data.

Turn 3 (window: 4 msgs, has summary: False):
  User: We handle about 500 tickets per day.
  AI:   With 500 tickets a day, set up a queue‑based architecture (e.g., RabbitMQ or AWS SQS) and use async workers to process intents, then route resolved tickets back to your ticketing system (Zendesk, Freshdesk, etc.) for logging and escalation.

Turn 4 (window: 2 msgs, has summary: True):
  User: Most questions are about billing and refunds.
  AI:   Train the bot on billing‑and‑refund intents and hook i

---

## 9. Choosing the Right Memory Type — Decision Framework

### Step-by-Step Decision

**Step 1 — Conversation Length:**
- 1–15 turns → Buffer Memory
- 15–50 turns → Window or Token Buffer
- 50+ turns → Summary Buffer

**Step 2 — Context Requirements:**
- Need exact recent messages? → Buffer, Window, or Summary Buffer
- Need old context preserved? → Summary, Summary Buffer, or Entity
- Need topic-based recall? → VectorStore

**Step 3 — Cost Constraints:**
- Minimal budget → Buffer or Window (no extra LLM calls)
- Moderate budget → Summary Buffer
- Generous budget → Entity or VectorStore

> **Default recommendation:** Start with **ConversationSummaryBufferMemory**. It handles the widest range of scenarios.

---

## 10. Persistent Memory & Advanced Patterns

All memory types above are **in-memory** — lost on restart. Production apps need persistence.

### Persistence Backends

| Backend | Use Case |
|---------|----------|
| **Redis** | Fast session-based memory, TTL support |
| **PostgreSQL** | Structured storage, ACID compliance |
| **MongoDB** | Flexible document storage |
| **SQLite** | Local dev/testing |

### Advanced Patterns

| Pattern | Description |
|---------|-------------|
| **Multi-Memory** | Window + Entity + VectorStore simultaneously |
| **Token Budget Mgmt** | Dynamically adjust memory to available context space |
| **Filtered Memory** | Remove PII, skip small talk, deduplicate |
| **Cross-Session** | Persist entity/vector memory across sessions for personalization |

### Memory Best Practices

| Practice | Description |
|----------|-------------|
| Start with Summary Buffer | Best default for most applications |
| Set explicit token budgets | Don't let memory grow unbounded |
| Use cheaper models for memory ops | Summary updates don't need your best model |
| Persist memory in production | In-memory is for development only |
| Reformulate queries in RAG | Follow-ups with pronouns will fail without it |
| Key memory by session ID | Never share memory across unrelated conversations |
| Test with long conversations | Memory bugs surface after many turns (50+) |

### Common Pitfalls

| Pitfall | Solution |
|---------|----------|
| Buffer for long chats | Switch to Summary Buffer |
| No reformulation in conv. RAG | Add question reformulation step |
| Ignoring token budgets | Set explicit `max_token_limit` |
| Same model for summaries | Use a smaller model for memory ops |
| No session isolation | Key memory by unique session ID |
| No persistence | Persist to Redis/PostgreSQL |

---

## 11. Sandbox — Try It Yourself!

In [ ]:
# ============================================================
#  SANDBOX - Try different memory types!
# ============================================================

# Choose a memory type
memory_type = "summary_buffer"  # Options: "buffer", "window", "summary_buffer"

# ============================================================

if memory_type == "buffer":
    mem = BufferMemory()
elif memory_type == "window":
    mem = WindowMemory(k=3)
elif memory_type == "summary_buffer":
    mem = SummaryBufferMemory(llm, max_recent=4)

my_question = "Tell me about Python programming."

if isinstance(mem, SummaryBufferMemory):
    data = mem.load()
    response = sb_chain.invoke({
        "summary": data["summary"],
        "recent": data["recent"],
        "input": my_question
    })
else:
    response = convo_chain.invoke({
        "history": mem.load(),
        "input": my_question
    })

mem.save(my_question, response)

print(f"[{memory_type.upper()}]")
print(f"Q: {my_question}")
print(f"A: {response}")

---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **LLMs are stateless** | Every API call is isolated — memory bridges this gap |
| **MessagesPlaceholder** | The standard way to inject history into prompts |
| **Buffer Memory** | Stores everything — simple, perfect recall, but overflows |
| **Window Memory** | Last K turns — fixed size, sharp "cliff" cutoff |
| **Token Buffer** | Like window but token-counted — precise budget control |
| **Summary Memory** | LLM-compressed — scales to long chats, costs extra calls |
| **Summary Buffer** | Verbatim recent + summarized old — **best general default** |
| **Entity Memory** | Tracks people/projects/tools — great for CRM-like apps |
| **VectorStore Memory** | Semantic similarity retrieval — best for very long chats |
| **Question Reformulation** | **Mandatory** for conversational RAG — pronouns break retrieval |
| **Token Budgets** | Memory, retrieval, system prompt, response all compete for context |
| **Persistence** | In-memory is dev only — use Redis/PostgreSQL in production |
| **Default Choice** | Start with Summary Buffer, specialize only if needed |